# Preprocessing 2: Train, validation and test split
**Input:** the clean posts from notebook 1. **Output:** the same posts, each assigned to a split.

In [ ]:
# Make the shared code in src/ importable, wherever the repository folder is
import os, sys
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "src", "mvsa")) and root != os.path.dirname(root):
    root = os.path.dirname(root)
sys.path.insert(0, os.path.join(root, "src"))

from mvsa.preprocessing import *

df = load_state("posts_clean")
REPORT = load_state("report")

## Step 5: Train / validation / test split, *before* fitting anything
**Why split now?** Every step from here on *learns* from the data: TF-IDF learns a vocabulary, and the
scaler learns means and standard deviations. If those were fitted on all the data, information from the test
set would leak into training and the test score would be too optimistic. So we split first and fit only on `train`.

* **8:1:1, stratified**. Stratification is on *label × agreement group*, so each split has the same
  class balance **and** the same share of soft mismatches, which the per-group evaluation needs.
* **Grouped by cleaned text.** MVSA has retweets: the same caption appears with different images.
  `StratifiedGroupKFold` keeps identical captions in the same split, so the model can't score well just by
  remembering a caption it saw in training.
* Hard mismatches form a fourth, evaluation-only set: `hard`.

Sizes on the full data: 3,591 / 446 / 447.

In [7]:
# Hard mismatches have no single label, so they stay out of the three modelling splits
main = df[df["mismatch_type"] != "hard_mismatch"].reset_index(drop=True)
hard = df[df["mismatch_type"] == "hard_mismatch"].reset_index(drop=True)

def group_keys(frame):
    key = frame["text_clean"].str.lower().str.replace(r"\W+", " ", regex=True).str.strip()
    return np.where(frame["text_empty"] == 1, "__empty__" + frame["id"], key)   # empty captions aren't one group

# Stratify on label x agreement group, and group by caption so a retweeted caption can't cross splits
strata = (main["label"] + "|" + main["mismatch_type"]).values
groups = group_keys(main)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)       # only appears for tiny strata
    # 1 fold of 10 becomes test; then 1 of the remaining 9 becomes validation, giving 8:1:1
    rest_idx, test_idx = next(StratifiedGroupKFold(10, shuffle=True, random_state=SEED).split(main, strata, groups))
    tr, va = next(StratifiedGroupKFold(9, shuffle=True, random_state=SEED)
                  .split(rest_idx, strata[rest_idx], groups[rest_idx]))

main["split"] = "train"
main.loc[rest_idx[va], "split"] = "val"
main.loc[test_idx, "split"] = "test"
hard["split"] = "hard"

# Safety check: stop the notebook if any caption appears in two splits
g = pd.Series(groups)
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    overlap = set(g[main.split == a]) & set(g[main.split == b])
    assert not overlap, f"caption leak between {a} and {b}"
print("No caption appears in more than one split.\n")

summary = pd.concat([
    main.groupby("split").size().rename("posts"),
    pd.crosstab(main.split, main.label, normalize="index").mul(100).round(1).add_suffix(" %"),
    pd.crosstab(main.split, main.mismatch_type, normalize="index").mul(100).round(1).add_suffix(" %"),
], axis=1).loc[["train", "val", "test"]]
print(summary.to_string())
print(f"\nhard (evaluation only): {len(hard)} posts")
REPORT["split_sizes"] = {**main.split.value_counts().to_dict(), "hard": len(hard)}
tick("Step 5 (split)")

posts = pd.concat([main, hard], ignore_index=True)    # one table, one row order, used for every feature
SPLITS = ["train", "val", "test", "hard"]

No caption appears in more than one split.

       posts  negative %  neutral %  positive %  agreement %  soft_mismatch %
split                                                                        
train   3591        30.0       10.5        59.5         57.6             42.4
val      446        30.0       11.9        58.1         57.2             42.8
test     447        30.0        8.7        61.3         56.6             43.4

hard (evaluation only): 355 posts

⏱ Step 5 (split) took 4.9s


## Save for the next notebook

In [ ]:
save_state("posts_split", posts)
save_state("report", REPORT)